# Advanced IR: Word Embeddings (Part 2)

In the second part of this tutorial, you will play around with word embeddings from various language models; from BOW to word2vec to transformers.

In [ ]:
!pip install nltk scikit-learn transformers gensim

In [ ]:
import nltk
nltk.download("punkt_tab")
nltk.download("shakespeare")

from nltk.corpus import shakespeare
from xml.etree import ElementTree

from sklearn.feature_extraction.text import CountVectorizer
import gensim.downloader
from gensim.models import Word2Vec
from gensim.test.utils import common_texts

from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel, AutoTokenizer, AutoModelForCausalLM

### Step 1: BOW

Let's imagine we'd like to represent the words in titles of some plays from Shakespeare.

In [ ]:
documents = [
"All's Well That Ends Well",
"As You Like It",
"The Comedy of Errors",
"Cymbeline",
"Love's Labours Lost",
"Measure for Measure",
"The Merry Wives of Windsor",
"The Merchant of Venice",
"A Midsummer Night's Dream",
"Much Ado About Nothing",
"Pericles, Prince of Tyre",
"Taming of the Shrew",
"The Tempest",
"Troilus and Cressida",
"Twelfth Night",
"Two Gentlemen of Verona",
"Winter's Tale",
]

We can simply and very cheaply create our BOW representations from them.

In [ ]:
vectoriser = CountVectorizer(lowercase=True,min_df=1)
model_bow = vectoriser.fit_transform(documents)
model_bow.toarray()

Notice how every document is encoded into a sparse representation. The representation of a word corresponds to its index in the sparse embedding.

We can see this representation signified by the values in this dictionary:

In [ ]:
vectoriser.vocabulary_

### Step 2: word2vec

So what it we wanted a bit of a more complex representation for words.

Well, first we need to tokenise our documents into individual tokens.

In [ ]:
doc_tokens = [nltk.word_tokenize(d.lower()) for d in documents]
doc_tokens

Then we can feed these tokens into our word2vec model to train them.

In [ ]:
model_w2v = Word2Vec(sentences=doc_tokens, vector_size=64, window=5, min_count=1, workers=5)
model_w2v.train(doc_tokens, total_examples=len(documents)+len(common_texts), epochs=100)

Once we've trained the model, we can inspect the vector for each word. 

In [ ]:
model_w2v.wv['comedy']

What happens when you request a word not in those documents?

Since each word is represented as a vector, we can measure the distance between it and other words in the vocabulary.

In [ ]:
model_w2v.wv.most_similar("comedy")

What do you think? Does this look reasonable? 

Probably not. That's because there wasn't much data to train anything from.

Let's train a new model on the entire play of romeo and juliet.

In [ ]:
play = ["".join(list(p.itertext())).lower() for p in shakespeare.xml('r_and_j.xml')]
play

In [ ]:
play_tokens = [nltk.word_tokenize(d.lower()) for d in play]
play_tokens

In [ ]:
model_w2v = Word2Vec(sentences=play_tokens, vector_size=128, window=12, min_count=1, workers=5)
model_w2v.train(play_tokens, total_examples=len(play_tokens), epochs=800)

So, now we have our trained model, let's look at some similar words.

In [ ]:
model_w2v.wv.most_similar("benvolio")

In [ ]:
model_w2v.wv.most_similar("dagger")

In [ ]:
model_w2v.wv.most_similar("fire")

Feel free to spend some time playing with these word representations. What do you think about them? Are they better than the ones trained with very little data?

What about a model trained on much, much more data?

In [ ]:
model_w2v_pretrained = gensim.downloader.load("word2vec-google-news-300")

Let's compare the similar words with our small model from above.

In [ ]:
model_w2v_pretrained.most_similar("dagger")

In [ ]:
model_w2v_pretrained.most_similar("fire")

We can even do some interesting things, like finding the words that are most similar to two other words (i.e., that kings and queens are both monarchs).

In [ ]:
model_w2v_pretrained.most_similar(["king", "queen"])

### Step 3: Transformers

We saw in the text ranking representations lecture that transformers are able to represent words using context. So let's investigate this claim deeper.

In [ ]:
tokeniser = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2")

First of all, let's look at the vocabulary of the transformer. The strange character at the start of a token means that is a sub-token. The tokeniser will split words that are not in the vocabulary into smaller and smaller sub-tokens until the original word is represented as a sequence of sub-tokens.

In [ ]:
tokeniser.get_vocab()

Next, we can see the input embeddings that the transformer model maps the ids to. This massive matrix can be indexed by the vocabulary id. Each row in this matrix (input representation) is then used to compute the K,Q, and V representation of a token.

In [ ]:
embedding_matrix = model.wte.weight.detach().cpu().numpy()
embedding_matrix

Let's try to find the token id for a single word.

In [ ]:
token_id = tokeniser.encode("fish", add_special_tokens=False)
token_id

And we can use that to look up the corresponding input representation. This is equivalent to the word representation learnt by word2vec.

In [ ]:
embedding_matrix[token_id[0]]

But uncommon words are tokenised into subtokens: look at the word `romeo`.

In [ ]:
token_ids = tokeniser.encode("romeo", add_special_tokens=False)
token_ids

We can feed all the tokens through the model to obtain our contextualised representations of the sequence:

In [ ]:
model(**tokeniser("romeo", return_tensors="pt"))

But GPT2 is also trained as a decoder, to predict the next token given a sequence, otherwise known as causal language modelling.

In [ ]:
model_causal = GPT2LMHeadModel.from_pretrained("gpt2")

With this kind of model, we can make a prediction about the next token in the sequence by modelling the task as an extreme classification problem. The logits returned here represent the raw prediction values from the model, scoring every token in the vocabulary. The token with the highest score is the most likely next token.

In [ ]:
model_causal(**tokeniser("fish", return_tensors="pt")).logits

We can use this idea to auto-regressively predict the next token in a sequence to generate entire sequences.

In [ ]:
outputs = model_causal.generate(**tokeniser("romeo: ", return_tensors="pt"), max_new_tokens=16)
tokeniser.decode(outputs[0], skip_special_tokens=True)

The results aren't very good because the model was not trained on as much data as newer models and doesn't have as many parameters.

But we can try out one of these newer models to see just how much better it is at generating text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
prompt = "Who were the characters in romeo and juliet?"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
print(tokenizer.decode(output_ids[0:], skip_special_tokens=True))

### Step 4: Sequence Representations

None of the above representations so far provide an encoding for a __sequence__ of text. How do you think you would derive a representation for the overall sequence rather than the individual tokens in the sequence? 

**HINT** the bi-encoder and cross-encoder models already compute sequences for queries and documents. Take a look in this library to try to figure out how these models compute a single representation for text.